# Transformers and LLMs — Practical Examples
**CubicNova Data Science Roadmap · Module 10.07**

This notebook explains tokenization, embeddings, self-attention, causal masks, transformer blocks, generation controls, and optional inference with Hugging Face Transformers.

The first examples are small and run locally. The pretrained-model section may download model files and needs internet access.

## 0. Imports and reproducibility

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 1. Tokenization: a toy example

In [ ]:
vocab = {
    "<unk>": 0,
    "transformers": 1,
    "learn": 2,
    "from": 3,
    "context": 4,
    ".": 5,
}
sentence = "Transformers learn from context .".lower().split()
token_ids = [vocab.get(token, vocab["<unk>"]) for token in sentence]
print("Tokens:", sentence)
print("Token IDs:", token_ids)

This deliberately simple tokenizer uses whitespace splitting and a tiny vocabulary. Real tokenizers use subword units, normalization rules, and special tokens.

## 2. Embeddings and positional information

In [ ]:
embedding = nn.Embedding(num_embeddings=20, embedding_dim=8)
ids = torch.tensor([[1, 2, 3, 4]])
token_vectors = embedding(ids)
print("Token ID shape:", ids.shape)
print("Embedding shape:", token_vectors.shape)  # batch, sequence, embedding dimension

def sinusoidal_positions(length, dim):
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    dims = torch.arange(dim, dtype=torch.float32).unsqueeze(0)
    rates = torch.pow(10000.0, -(2 * torch.floor(dims / 2)) / dim)
    angles = positions * rates
    enc = torch.zeros(length, dim)
    enc[:, 0::2] = torch.sin(angles[:, 0::2])
    enc[:, 1::2] = torch.cos(angles[:, 1::2])
    return enc

pos = sinusoidal_positions(length=4, dim=8)
print("Position encoding shape:", pos.shape)
combined = token_vectors + pos.unsqueeze(0)
print("Combined shape:", combined.shape)

Embeddings represent token IDs as vectors. Positional information gives the model a representation of token order. This sinusoidal example is for teaching; many modern architectures use learned or other positional schemes.

## 3. Scaled dot-product self-attention

In [ ]:
# One batch, 3 tokens, 4 features
X = torch.tensor([[
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0],
]])

# Simplified example: use X as Q, K, and V to focus on the attention calculation
Q, K, V = X, X, X
scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
weights = torch.softmax(scores, dim=-1)
output = weights @ V

print("Scores:\n", scores)
print("Attention weights:\n", weights)
print("Row sums:", weights.sum(dim=-1))
print("Output shape:", output.shape)

plt.figure(figsize=(4, 3))
plt.imshow(weights[0].numpy(), vmin=0, vmax=1)
plt.colorbar(label="Attention weight")
plt.xlabel("Key token")
plt.ylabel("Query token")
plt.title("Toy self-attention weights")
plt.show()

In a learned attention layer, Q, K, and V are separate learned projections of the input. Here they are tied to keep the calculation visible.

## 4. Causal attention mask

In [ ]:
length = 5
scores = torch.randn(length, length)
future_mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
causal_scores = scores.masked_fill(future_mask, float("-inf"))
causal_weights = torch.softmax(causal_scores, dim=-1)

print("Future positions have zero attention:")
print(causal_weights)
assert torch.allclose(
    causal_weights.masked_select(future_mask),
    torch.zeros_like(causal_weights.masked_select(future_mask))
)

plt.figure(figsize=(4, 3))
plt.imshow(causal_weights.numpy(), vmin=0, vmax=1)
plt.colorbar(label="Attention weight")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Causal attention")
plt.show()

A causal mask prevents a token from attending to future positions. A padding mask serves a different purpose: it hides artificial padding tokens.

## 5. A simple attention module

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, dim, causal=False):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.causal = causal
        self.scale = math.sqrt(dim)

    def forward(self, x):
        # x: batch, sequence, dim
        Q, K, V = self.q(x), self.k(x), self.v(x)
        scores = Q @ K.transpose(-2, -1) / self.scale
        if self.causal:
            n = x.shape[1]
            mask = torch.triu(torch.ones(n, n, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V

demo = torch.randn(2, 6, 16)
attention = SelfAttention(dim=16, causal=True)
result = attention(demo)
print("Input:", demo.shape, "Output:", result.shape)

This is a minimal teaching layer, not a production implementation. Multi-head attention splits projections into multiple heads and real models include additional architectural details.

## 6. A compact transformer block

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, dim, hidden_dim, causal=True):
        super().__init__()
        self.attention = SelfAttention(dim, causal=causal)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        x = self.norm1(x + self.attention(x))  # attention + residual
        x = self.norm2(x + self.ffn(x))        # feed-forward + residual
        return x

block = TinyTransformerBlock(dim=16, hidden_dim=64)
out = block(torch.randn(2, 5, 16))
print("Block output shape:", out.shape)

Residual connections add the sublayer input back to its output. Layer normalization stabilizes representations. Transformer implementations vary in normalization order and attention details.

## 7. Generation controls: greedy choice and temperature

In [ ]:
def temperature_distribution(logits, temperature=1.0):
    scaled = logits / temperature
    return torch.softmax(scaled, dim=-1)

next_token_logits = torch.tensor([1.0, 2.0, 0.5, 3.0])
for temp in (0.5, 1.0, 2.0):
    probs = temperature_distribution(next_token_logits, temperature=temp)
    print(f"temperature={temp}: probabilities={probs.numpy().round(3)}")

greedy_id = int(torch.argmax(next_token_logits))
print("Greedy next-token ID:", greedy_id)

Lower temperatures sharpen a distribution and higher temperatures flatten it. Greedy decoding selects the maximum logit; sampling can produce variation. These controls do not guarantee factuality.

## 8. Optional: Hugging Face inference

This section may download model files and requires internet access. Use only a model/checkpoint you are permitted to use, and check its license and intended use.

The example uses a small text-generation checkpoint. If loading fails because of network restrictions, memory limits, or a changed model repository, skip this section and continue with the conceptual exercises.

In [ ]:
# Optional section — uncomment to run if Transformers is installed and downloads are allowed.
#
# from transformers import pipeline
# generator = pipeline(
#     "text-generation",
#     model="distilgpt2",
#     device=0 if torch.cuda.is_available() else -1,
# )
# prompt = "A neural network learns patterns by"
# result = generator(
#     prompt,
#     max_new_tokens=40,
#     do_sample=False,
#     pad_token_id=generator.tokenizer.eos_token_id,
# )
# print(result[0]["generated_text"])

For reproducibility, record the model ID/revision, tokenizer, package versions, prompt, generation parameters, and the date/model files used. Treat outputs as candidates to verify, not authoritative facts.

## 9. Evaluation and responsible-use checklist

For a language-model experiment, evaluate:

- **Task fit:** does the output answer the intended question?
- **Factuality:** can claims be checked against reliable sources?
- **Coverage:** are required points omitted?
- **Consistency:** does behavior remain stable across paraphrases?
- **Safety and bias:** are there harmful or unfair outputs?
- **Privacy:** did the prompt include data that should not be shared?
- **Prompt injection:** can untrusted text manipulate the system or tool use?
- **Efficiency:** latency, token count, memory, and cost.

For important use cases, use a representative test set, human review where appropriate, and explicit acceptance criteria.

## 10. Experiment checklist

1. Change attention input values and inspect attention weights.
2. Compare causal and non-causal attention.
3. Compare temperatures `0.5`, `1.0`, and `2.0`.
4. If using a pretrained model, compare greedy decoding with sampling on the same prompts.
5. Document at least five failure cases and explain their impact.

Do not select the model solely from a few appealing outputs. Keep the evaluation method fixed and record all settings.

## Key takeaways

- Tokens are converted to IDs and embeddings; positional information represents order.
- Self-attention combines values according to query-key compatibility.
- Causal masks prevent access to future tokens in autoregressive models.
- Transformer blocks combine attention, residual connections, normalization, and feed-forward layers.
- Text generation is probabilistic and must be evaluated for factuality, safety, and task fit.